In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

ALPHA, POWER = 0.05, 0.80
z = norm.ppf(1 - ALPHA / 2) + norm.ppf(POWER)

def n_per_arm(p1, rel_lift):
    p2 = p1 * (1 + rel_lift)
    return int(np.ceil(z**2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2))

rows = []
for p1 in [0.02, 0.05, 0.10]:
    for lift in [0.05, 0.10, 0.14]:
        rows.append({"baseline_rate": f"{p1:.0%}", "relative_lift": f"{lift:.0%}",
                     "users_per_arm": n_per_arm(p1, lift),
                     "total_users": 2 * n_per_arm(p1, lift)})
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nz = {z:.4f}  (alpha {ALPHA}, two-sided, power {POWER})")

baseline_rate relative_lift  users_per_arm  total_users
           2%            5%         315204       630408
           2%           10%          80679       161358
           2%           14%          41928        83856
           5%            5%         122121       244242
           5%           10%          31231        62462
           5%           14%          16219        32438
          10%            5%          57760       115520
          10%           10%          14749        29498
          10%           14%           7649        15298

z = 2.8016  (alpha 0.05, two-sided, power 0.8)


## A/B test plan (design only, not run)

My offline results say the LightGBM ranker beats my best simple method by about 14% on MAP@12. That's one past week, scored against what people happened to buy. It doesn't prove the ranker would make people buy more. The only way to know that is a live test, so here is how I'd set one up.

**What's compared.** Group A gets the current method (past items, bought-together, then best-sellers). Group B gets the LightGBM ranker. Everything else on the page stays the same.

**How users are split.** 50/50 at random, by customer ID, so the same person always sees the same version for the whole test. New customers with no history go through the same split, even though both groups would mostly show them best-sellers.

**Main metric.** The share of users who buy at least one recommended item within 7 days of seeing it. I picked this over MAP@12 because it's closer to what the business cares about, and over clicks because clicks don't pay.

**Guardrails.** Revenue per user, return rate and page load time. If B wins on the main metric but people return more or the page slows down, it isn't a real win.

**How many users.** I don't know H&M's real conversion rate from recommendations, so I worked it out for a few possible rates (5% significance, 80% power, two-sided):

| Baseline rate | Lift to detect | Total users needed |
|---|---|---|
| 2% | 10% | 161,358 |
| 5% | 10% | 62,462 |
| 5% | 5% | 244,242 |
| 10% | 10% | 29,498 |

I'd plan for a 10% lift, not the 14% I saw offline. Offline gains usually shrink in real life, and planning for the smaller effect means the test is less likely to miss a real but modest win.

**How long.** At least two full weeks, even if the sample size is reached sooner. My drift check showed that results change a lot from week to week, so one week could just be a lucky or unlucky week. Two weeks also covers every weekday twice. I'd decide the end date before starting and not stop early just because B looks ahead. Checking every day and stopping at the first good result makes false wins much more likely.

**What would make me ship it.** B is better on the main metric with a 95% confidence interval that doesn't include zero, and no guardrail gets meaningfully worse.